In [ ]:
# ============================================
# scRNA-seq basic Scanpy workflow
# ============================================
#
# This notebook:
# 1. Loads 10x matrix.mtx.gz datasets
# 2. Merges 6 mtxitions
# 3. Performs preprocessing
# 4. Computes PCA + UMAP
# 5. Plots UMAP colored by mtxition
#
# Required directory structure:
#
# ============================================
# 1. Install packages (run once)
# ============================================

%pip install scanpy anndata matplotlib leidenalg

In [ ]:
# ============================================
# Download data from GEO
# ============================================ 
!rm -rf data
!mkdir data
!wget "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE196280&format=file" -O GSE196280.tar
!tar -xvf GSE196280.tar --directory data/

In [ ]:

# ============================================
# 2. Imports
# ============================================

import scanpy as sc
import re
import anndata as ad
import pandas as pd
import numpy as np
from pathlib import Path


In [ ]:
# Plot settings
sc.settings.verbosity = 3
sc.set_figure_params(figsize=(6, 6))

base = Path("data/")
adatas = []
#search all matrix.mtx.gz files in data/ and remove the matrix.mtx.gz suffix to get the mtxition name
for mtx in base.iterdir():
    if not mtx.is_file() or not mtx.name.endswith("matrix.mtx.gz"):
        continue

    mtxition = mtx.name.replace(".matrix.mtx.gz", "")

    #create a subdir called mtxition and move all files with the same mtxition name into it
    dir = base / mtxition
    dir.mkdir(exist_ok=True)
    for file in base.iterdir():
        if file.is_file() and file.name.startswith(mtxition):
            file.rename(dir / file.name.replace(mtxition+".", ""))


for dir in base.iterdir():
    #filter for "naive" in name
    if not dir.is_dir() or "naive" not in dir.name.lower():
        continue

    condition = re.sub(r"GSM[^_]+_", "", dir.name)
    print(f"Loading {condition} from {dir}...")

    adata = sc.read_10x_mtx(
        dir,
        var_names="gene_symbols",
        cache=True
    )

    # Add metadata
    adata.obs["name"] = condition

    # Make unique cell IDs
    adata.obs_names_make_unique()

    # Optional: prefix barcodes with mtxition
    adata.obs_names = [
        f"{condition}_{cell}"
        for cell in adata.obs_names
    ]

    adatas.append(adata)


In [ ]:
# ============================================
# 5. Merge datasets
# ============================================
print(adatas)

adata = ad.concat(
    adatas,
    axis=0,
    join="outer",
    label="name",
    keys=[adata.obs["name"][0] for adata in adatas],
    index_unique="-"
)

print(adata)

In [ ]:
# ============================================
# 6. Basic QC metrics
# ============================================

# Mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")

# Calculate QC metrics
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    percent_top=None,
    log1p=False,
    inplace=True
)

# View summary
adata.obs.head()


In [ ]:
# ============================================
# 7. QC plots
# ============================================

sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True
)


In [ ]:

sc.pl.scatter(
    adata,
    x="total_counts",
    y="n_genes_by_counts",
    color="name"
)


In [ ]:
# just checking if some interesting genes are present in the dataset
marker_genes = [
    "SOX2",    
    "GATA3",  
    "TMEM88",    
    "CLDN4",
    "CD24",
    "NANOG" 
]

existing_markers = [
    g for g in marker_genes
    if g in adata.var_names
]

print(existing_markers)

In [ ]:

# ============================================
# 8. Filter low-quality cells/genes
# ============================================

# Filter cells
sc.pp.filter_cells(adata, min_genes=200)

# Filter genes
sc.pp.filter_genes(adata, min_cells=3)

# Remove high mitochondrial cells
adata = adata[adata.obs.pct_counts_mt < 15].copy()

print(adata)


In [ ]:
# ============================================
# 9. Normalize and log transform
# ============================================

# Save raw counts
adata.layers["counts"] = adata.X.copy()

# Normalize counts per cell
sc.pp.normalize_total(
    adata,
    target_sum=1e4
)

# Log transform
sc.pp.log1p(adata)


In [ ]:
# ============================================
# 10. Highly variable genes
# ============================================

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    subset=False, #want to keep all the genes for later visualization
    flavor="seurat"
)

adata.raw = adata
adata = adata[:, adata.var.highly_variable].copy()

# Plot HVGs
sc.pl.highly_variable_genes(adata)

In [ ]:
# ============================================
# 11. Scale data
# ============================================

sc.pp.scale(
    adata,
    max_value=10
)

In [ ]:
# ============================================
# 12. PCA
# ============================================

sc.tl.pca(
    adata,
    svd_solver="arpack"
)

# PCA variance plot
sc.pl.pca_variance_ratio(
    adata,
    log=True
)

In [ ]:
sc.pl.pca(
    adata,
    color="name",
    components=["1,2", "2,3"]
)

sc.pl.pca(
    adata,
    color="name",
    components=["3,4", "4,5"]
)



In [ ]:
# ============================================
# 13. Compute neighbors
# ============================================

sc.pp.neighbors(
    adata,
    n_neighbors=15,
    n_pcs=30
)


# ============================================
# 14. UMAP
# ============================================

sc.tl.umap(adata)


In [ ]:
# ============================================
# 15. Leiden clustering
# ============================================

sc.tl.leiden(
    adata,
    resolution=0.5
)

In [ ]:
# ============================================
# 16. Plot UMAPs
# ============================================

# Colored by mtxition
sc.pl.umap(
    adata,
    color="name",
    frameon=False
)

# Colored by cluster
sc.pl.umap(
    adata,
    color="leiden",
    legend_loc="on data",
    frameon=False
)

In [ ]:
# ============================================
# 17. Marker gene visualization
# ============================================

#print all available adata.var_names

marker_genes = [
    "SOX2",  
    "NANOG",
    "GATA3", 
    "CLDN4",
    "TP63",
    "TMEM88", 
    "KRT18",  
    "TBXT",
    "SOX9",
    "HAND1",
    "TWIST1",
    "TBX3",
    "CDX1",
    "CDX2",
    "TFAP2A",
    "TFAP2C", 
    "CD24", 
]


In [ ]:

existing_markers = [
    g for g in marker_genes
   if g in adata.var_names
]

print(existing_markers)

# plot panel with 2 plots per row
sc.pl.umap(
    adata,
    color=existing_markers,
    ncols=3
)

In [ ]:


# ============================================
# 18. Find marker genes per cluster
# ============================================

sc.tl.rank_genes_groups(
    adata,
    groupby="leiden",
    method="wilcoxon"
)

# Show top markers
sc.pl.rank_genes_groups(
    adata,
    n_genes=20,
    sharey=False
)



In [ ]:
# PAGA trajectory graph
sc.tl.paga(
    adata,
    groups="leiden"
)

# Plot connectivity graph
sc.pl.paga(
    adata,
    threshold=0.03,
    show=True
)

In [ ]:
sc.tl.umap(
    adata,
    init_pos='paga'
)

sc.pl.umap(
    adata,
    color=["leiden", "name"]
)

In [ ]:
adata.uns['iroot'] = np.flatnonzero(
    adata.obs['leiden'] == '0'
)[0]

sc.tl.dpt(adata)

sc.pl.umap(
    adata,
    color="dpt_pseudotime",
    cmap="viridis"
)

In [ ]:
sc.pl.umap(
    adata,
    color=["name", "dpt_pseudotime"],
    wspace=0.4
)

In [ ]:
sc.pl.paga_compare(
    adata,
    basis='umap',
    show=True
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Convert obs to dataframe
df = adata.obs.copy()

plt.figure(figsize=(8, 5))

sns.kdeplot(
    data=df,
    x="dpt_pseudotime",
    hue="name",
    fill=True,
    common_norm=False,
    alpha=0.4
)

plt.xlabel("Pseudotime")
plt.ylabel("Density")
plt.title("Cell density along pseudotime")
plt.show()

In [ ]:
%pip install cellrank

import cellrank as cr
from cellrank.kernels import ConnectivityKernel
from cellrank.estimators import GPCCA
import matplotlib.pyplot as plt

# ----------------------------------------------------
# ASSUMPTION:
# You already ran:
# sc.tl.dpt(adata)
# and have:
# adata.obsm["X_pca"], neighbors, UMAP, etc.
# ----------------------------------------------------


# ====================================================
# 1. Ensure neighbors exist (required for CellRank)
# ====================================================

#sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)


# ====================================================
# 2. Build diffusion-based kernel (no RNA velocity needed)
# ====================================================

ck = ConnectivityKernel(adata)
ck.compute_transition_matrix()


# ====================================================
# 3. Learn global fate structure (macrostates)
# ====================================================

g = GPCCA(ck)

# number of macrostates = adjust based on biology
# (try 3–6 usually)
g.compute_schur()

In [ ]:
g.compute_macrostates(n_states=4)

g.plot_macrostates(which="all")
plt.show()

In [ ]:
# Alternative Scanpy view:
g.predict_terminal_states()
g.compute_fate_probabilities()

g.plot_fate_probabilities(basis="umap")
plt.show()


In [ ]:
g.compute_lineage_drivers()
g.plot_lineage_drivers(lineage=0)
g.plot_lineage_drivers(lineage=1)
g.plot_lineage_drivers(lineage=2)


In [ ]:
g.compute_lineage_priming()
g.plot_macrostate_composition(key='name')

In [ ]:
sc.pl.dotplot(adata, marker_genes, groupby="leiden", standard_scale="var")

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata, groupby="leiden", standard_scale="var", n_genes=5)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata, groupby="leiden", standard_scale="var", n_genes=6)

In [ ]:
print(adata.obs.dtypes)

In [ ]:

# ============================================
# 19. Save processed object
# ============================================

adata.write("combined_scRNAseq.h5ad")

print("\nSaved combined_scRNAseq.h5ad")


# ============================================
# 20. Reload later
# ============================================

# adata = sc.read_h5ad("combined_scRNAseq.h5ad")

In [ ]:


# ============================================
# 20. Reload later
# ============================================

#adata = sc.read_h5ad("combined_scRNAseq.h5ad")